# 05 — Final Data Prep for Tableau

This is our final step! We're going to take our clean, analyzed data and package it up into a master CSV file that's ready to be plugged into Tableau (or any other BI tool).

**What we're doing here:**
1. **Benchmarking**: Calculating how each listing's price compares to its neighborhood average.
2. **KPIs**: Creating some final metrics like "Demand Score" and "Activity Status."
3. **Formatting**: Renaming columns so they look good in a dashboard.

### Setup

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Paths
project_root = Path.cwd().resolve().parent
input_path = project_root / 'data' / 'processed' / 'airbnb_nyc_cleaned.csv'
output_path = project_root / 'data' / 'processed' / 'nyc_airbnb_tableau_master.csv'

# Load data
df = pd.read_csv(input_path)
print(f"Loaded {len(df)} listings for final prep.")

## 1. Neighborhood Benchmarking
We'll create a `price_index` to see if a listing is priced higher or lower than the median for its specific neighborhood. A value of 1.0 means it's right at the median.

In [ ]:
hood_medians = df.groupby(['neighbourhood_group', 'neighbourhood'])['price'].transform('median')
df['price_index'] = (df['price'] / hood_medians).round(2)

print("Neighborhood price index calculated.")

## 2. Activity & Demand Metrics
Let's create a couple of final scores to help us filter the data in Tableau.

In [ ]:
# Demand Score: Combination of review rate and availability
df['demand_score'] = (df['review_rate_month'] * (df['availability_365'] / 365.0)).round(4)

# Activity Status: Is this listing currently active?
def get_status(year):
    if pd.isna(year): return 'Inactive (No Reviews)'
    if year == 2019: return 'Active (2019)'
    return 'Older Activity'

df['activity_status'] = df['review_year'].apply(get_status)

print("Demand and activity metrics generated.")

## 3. Renaming Columns for Visualization
Finally, we'll rename our columns to more readable names (e.g., `id` -> `Listing ID`) so they look professional in Tableau.

In [ ]:
column_map = {
    'id': 'Listing ID',
    'name': 'Listing Title',
    'host_id': 'Host ID',
    'host_name': 'Host Name',
    'neighbourhood_group': 'Borough',
    'neighbourhood': 'Neighborhood',
    'latitude': 'Latitude',
    'longitude': 'Longitude',
    'room_type': 'Room Type',
    'price': 'Price',
    'minimum_nights': 'Min Nights',
    'review_count': 'Total Reviews',
    'last_review': 'Last Review Date',
    'review_rate_month': 'Reviews per Month',
    'host_listing_count': 'Host Portfolio Size',
    'availability_365': 'Availability 365',
    'revenue_proxy': 'Annual Revenue Proxy',
    'price_index': 'Market Price Index',
    'demand_score': 'Listing Demand Score',
    'activity_status': 'Activity Status',
    'price_tier': 'Price Category'
}

df_final = df.rename(columns=column_map)

# Select only the columns we need for the dashboard
cols_to_keep = [c for c in column_map.values() if c in df_final.columns]
df_final = df_final[cols_to_keep]

df_final.to_csv(output_path, index=False)
print(f"Done! The final dataset has been saved to: {output_path}")